# P14 — Prédiction du taux de réussite au bac des lycées

**Contexte métier.** Rektorats (DANE), parents d'élèves, plates-formes de comparaison : expliquer les différences de taux de réussite entre lycées (au-delà de la valeur ajoutée) pour cibler les accompagnements. Enjeu : inégalités territoriales (Mayotte 75 % vs Paris > 96 %).

**Objectif de l'EDA.** Identifier la cible (taux de réussite toutes séries) et les régresseurs (secteur, académie, taille, mixité des séries).

**Données.** Indicateurs de résultat des lycées d'enseignement général et technologique — Ministère de l'Éducation nationale (data.education.gouv.fr). Licence ouverte Etalab 2.0. 32 485 lignes (lycée × année), 2 546 lycées, 2012-2025.

## 1. Téléchargement

Export CSV de l'open data Éducation nationale :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-indicateurs-de-resultat-des-lycees-gt_v2/exports/csv"
if not os.path.exists("data/indicateurs-lycees-gt.csv"):
    urllib.request.urlretrieve(URL, "data/indicateurs-lycees-gt.csv")
print(os.path.getsize("data/indicateurs-lycees-gt.csv") / 1e6, "Mo")

## 2. Chargement

Séparateur `;`, décimales à virgule, taux avec % : nettoyage numérique, calcul des parts de chaque série dans les présents.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/indicateurs-lycees-gt.csv", sep=";", low_memory=False, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
TR = "Taux de réussite - Toutes séries"
df[TR] = pd.to_numeric(df[TR].astype(str).str.replace("%", "").str.replace(",", "."), errors="coerce")
pres = [c for c in df.columns if c.startswith("Présents - ")]
for c in pres:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "."), errors="coerce")
for c in pres:
    df["part_" + c.replace("Présents - ", "")] = df[c] / df["Présents - Toutes séries"].replace(0, np.nan)
df["taille"] = df["Présents - Toutes séries"]
dfa = df.dropna(subset=[TR])
print(len(dfa), "lycées-années avec taux de réussite")

## 3. Analyse

Taux de réussite par secteur, par académie, par taille, par mixité de séries ; corrélation des régresseurs.

In [ ]:
print("Taux de réussite :", dfa[TR].describe(percentiles=[.1, .5, .9]).round(1).to_dict())
print()
print("Par secteur :", dfa.groupby("Secteur")[TR].agg(["median", "count"]).round(1).to_dict("index"))
print()
print("Par académie (bas) :", dfa.groupby("Académie")[TR].median().round(1).sort_values().head(5).to_dict())
print("Par académie (haut) :", dfa.groupby("Académie")[TR].median().round(1).sort_values().tail(5).to_dict())
parts = [c for c in dfa.columns if c.startswith("part_") and c != "part_Toutes séries"]
c = dfa[[TR, "taille"] + parts].apply(lambda s: pd.to_numeric(s, errors="coerce"))
print()
print("Corr. Spearman avec le taux :", c.corr("spearman")[TR].round(2).dropna().to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n lignes : 32485
- annees : 2012 → 2013 → 2014 → 2015 → 2016 → 2017 → 2018 → 2019 → 2020 → 2021 → 2022 → 2023 → 2024 → 2025
- n lycees : 2546
- taux reussite stats : count : 32 485.00, mean : 93.60, std : 6.30, min : 0.00, 10% : 86.00, 25% : 91.00, 50% : 95.00, 75% : 98.00, 90% : 10…
- taux reussite par secteur : privé sous contrat : median : 99.00, count : 10949, public : median : 93.00, count : 21536
- taux reussite par academie : MAYOTTE : 75.00, GUYANE : 88.00, LA REUNION : 93.00, AMIENS : 93.00, REIMS : 93.00, CRETEIL : 93.00, LILLE : 94.00, L…
- ecart prive public mediane : nan
- valeur ajoutee stats : count : 32 025.00, mean : -0.30, std : 3.70, min : -78.00, 10% : -4.00, 50% : 0.00, 90% : 3.00, max : 36.00
- part series top : L : 13.40, ES : 26.80, S : 39.70, Gnle : 76.20, STI2D : 25.00, STD2A : 28.40, STMG : 24.20, STL : 12.80, ST2S : 29.20…
- evolution reussite par annee : 2012 : 91.00, 2013 : 93.00, 2014 : 93.00, 2015 : 94.00, 2016 : 93.00, 2017 : 93.00, 2018 : 93.00, 2019 : 93.00, 2020 …
- missing pct : Taux de réussite - Toutes séries : 0.00, Secteur : 0.00, Académie : 0.00, taille_lycee : 0.00

## 5. Variable cible et régresseurs

**Cible (régression) : `Taux de réussite - Toutes séries`.** Médiane nationale 95 %, écart inter-lycée de 75 % (Mayotte) à 100 %. Variante : classification lycée "en difficulté" (< 85 %).

**Régresseurs retenus :**
- `Secteur` (public vs privé sous contrat) : écart médian de 6 points (93 vs 99) — à interpréter avec la sociologie, pas comme causal
- `Académie` / `Département` : facteur territorial fort
- `taille` (présents au bac) et mixité des séries (`part_STMG`, `part_STI2D`…) : la structure des séries pèse sur le taux
- `Année` (2012-2025, effet COVID 2020-2021)

**Pièges :**
- Les colonnes `Valeur ajoutée…` et `Taux d'accès 2nde-bac` sont des outputs calculés par l'administration À PARTIR de la cible : les inclure est une fuite de données — elles servent d'évaluation, pas de feature
- Taux = 0 pour des lignes sans candidats : exclure les lycées à très peu de présents (petites structures, taux volatils)
- Le privé sélectionne ses élèves : le modèle apprend un effet élève, pas un effet établissement — le dire dans la discussion
- Décimales à virgule et symbole % : nettoyage obligatoire